1. 核心零件:
- 多头注意力    Multi-Head Attention
- 掩码(防作弊)  Masking
- 前馈网络(消化吸收)    Feed forward
- 归一化与残差(稳定训练)    LayerNorm & Residuals

In [ ]:
# 把2.ipynb的核心框架拉过来拼在一起
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import matplotlib

class Block(nn.Module):
    '''一个完整的Transformer块'''
    def __init__(self, n_embd, n_head):
        '''
        n_embd: 词嵌入维度(Embedding Dimension)：每个词被转化成的向量长度
        n_head: 多头注意力的头数
        n_embd // n_head: 每个头的维度

        ln1, ln2: Layer Normalization 层归一化
        sa : Self-Attention 自注意力
        ffwd : Feed-Forward Network(有时也写FFN)前馈网络
        '''
        super().__init__()

        # 1. 层归一化
        self.ln1 = nn.LayerNorm(n_embd)
        # 2. 多头注意力(命名为雷达sa)， 通过Q/K/V, 根据与其他词的相关性重新计算自己的表达
        self.sa = MultiHeadAttention(n_head, n_embd // n_head)
        # 3. 前馈网络(消化)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd)

        # 多头注意力之前和前馈网络前都有一个LayerNorm, 
    
    def forward(self, x):
        # 残差连接： output = x + f(x)
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))

        return x
    
class NanoGPT(nn.Module):
    def forward(self, idx, targets=None):
        '''
        idx: 输入字符的ID序列
        target: 
        '''

        # 1. 加上位置编码，让模型了解顺序
        x = self.token_embedding_table(idx) + self.position_embedding_table

        # 2. 经过多层Transformer 块
        x = self.blocks(x)

        # 3. 归一化并输出概率
        logits = self.lm_head(self.ln1(x))

        # LM Head (Language Model Head): 这是一个线性层，它的作用是把隐藏层的向量投影到词表大小的空间。
        # Logits: 它的输出是一个分数值（Logits）。比如你的词表有 5000 个字，它就会输出 5000 个数。哪个数字最大，就代表模型认为下一个字最可能是谁。

        return logits

2. 组装之前：

训练大模型时，需遵循一个标准化循环：

“读取-计算-对答案-改错”.



In [ ]:
# 1. 数据加载：
def get_batch(split):
    # 根据是训练集还是验证集，取不同的数据
    data = train_data if split == 'train' else val_data
    
    # 随机产生 batch_size 个起始位置
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    # x 是输入，y 是对应的标签（即 x 后面那个字）
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    
    # 搬运到 GPU（如果有的话）
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
# 2. 核心训练循环，几乎所有深度学习模型的训练都是这样：

# 初始化优化器（就像是一个负责调螺丝的机械师）
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for iter in range(max_iters):
    # 1. 采样一波数据
    xb, yb = get_batch('train')

    # 2. 前向传播（让模型做题）
    logits, loss = model(xb, yb)

    # 3. 反向传播之前，先清空之前的“纠错记录”
    optimizer.zero_grad(set_to_none=True)

    # 4. 计算梯度（算出哪些参数该往哪边调）
    loss.backward()

    # 5. 更新参数（动手调螺丝）
    optimizer.step()

    # 定期打印进度
    if iter % eval_interval == 0:
        print(f"步数 {iter}: 损失值 (Loss) 为 {loss.item():.4f}")

注意：每个继承自nn.Module的类都要实现一个forward函数来处理数据。其特征和底层原理如下：

1. 魔法调用：在nn.Module底层实现了__call__魔法方法，当model = NanoGPT() (NanoGPT类继承自nn.Module)时，调用model(x)时，就会自动调用model.forward(self, x) （还有其他的一些内部维护工作等）
2. 计算图与自动微分的起点：当你执行一个运算时，Pytorch会自动记录运算双方，forward()则构建一个"计算图"来定义所有的偏导关系；最后调用loss.backward()时，就会沿着forward的反方向倒回来，自动计算出所有参数的梯度。
3. 它代表了类中的数据流向。以NanoGPT 的forward为例：


    class NanoGPT(nn.Module):
        def __init__(self):
            super().__init__()
            self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
            self.position_embedding_table = nn.Embedding(block_size, n_embd)
            self.blocks = nn.Sequential(*[Block(n_embd=n_embd, n_head=n_head) for _ in range(n_layer)])
            self.ln_f = nn.LayerNorm(n_embd)

            self.lm_head = nn.Linear(n_embd, vocab_size)

        def forward(self, idx, targets=None):
            B, T = idx.shape
            tok_emb = self.token_embedding_table(idx) # 拿到(batch_size, tokens, n_embd),即原词表中提取的部分特征向量
            # torch.arange(T) ：生成一个从 0 到 T-1 的等差数列张量
            # device = device : 在Pytorch中，两个张量必须处在同一个设备上才能进行运算
            pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (tokens, n_embd)
            x = tok_emb + pos_emb # 加上位置张量，给输入增添了位置信息(运算采用广播机制)
            x = self.blocks(x)
            x = self.ln_f(x)
            # 解码，输出的logits(batch_size, tokens, vocab_size)
            # 对每一批(batch)的每个词(tokens), 长度为vocab_size 的向量表示词表中"下一个词"出现的概率
            logits = self.lm_head(x)

            if targets is None: # target 标准答案 (B, T)
                loss = None
            else:
                B, T, C = logits.shape
                logits = logits.view(B*T, C)
                targets = targets.view(B*T)
                # F.cross_entropy的输入是二维的：(样本数，类别数)
                loss = F.cross_entropy(logits, targets) # 计算交叉熵

            return logits, loss

        def generate(self, idx, max_new_tokens):
    ......

- 在NanoGPT中，数据先从nn.Embedding(idx) (形状为(vocab_size, n_embd)的词表)拿到所选取的样本的特征向量: (batch_size, block_size, n_embd)
- 再从 nn.Embedding (这次是形状为 (block_size, n_embd) 的位置表) 拿到对应位置的特征向量， 并与样本特征向量合并(相加)得到输入x， 即将位置的影响因素也考虑进来
- 随后 将x通过 transformer块(self.blocks(x)),  层归一化(self.ln_f(x)), 根据每个词的特征向量为下一个词(共vocab_size个词)的可能性打分(self.lm_head(x)) 得到逻辑logits, 代表着预测结果
- 最后将预测结果与输入的target标准输出做比较，并计算交叉熵衡量loss
- 可以知道随后还会在forward外调用loss.backward(), 这时就会沿着forward的反方向传播梯度并更新每一层的参数。